# Debug: #619 split 3/3 — gap-edge formula change

Verifies the deferred patch in `compute_plausible_gaps`:

```python
# before (current master):
h_textlines[i].x0 - h_textlines[i - 1].x0   # left-edge-to-left-edge stride
# after:
h_textlines[i].x0 - h_textlines[i - 1].x1   # actual whitespace gap
```

**Setup:** `pip install -e .[plot]` in the branch worktree, then run this notebook.

**Expected outcome of this branch:** the xfailed tests in `tests/test_network.py` (test_issue_585, test_issue_585_network_flavor_with_table_areas) flip from `xfail` → `xpass`. No other test fixtures change.

**Reproducers used:**
- `tests/files/multiple_tables.pdf` (#585 original report)
- `tests/files/good_energy.pdf` (added in #745)

Issue: [#770](https://github.com/camelot-dev/camelot/issues/770) · Tracking PR (TODO marker only): #771

In [ ]:
# 1. Confirm we're on the debug branch + see the patch site
import subprocess, sys, os
os.chdir(os.path.dirname(os.path.dirname(os.path.abspath('.'))) if 'notebooks' in os.getcwd() else '.')
print('branch:', subprocess.check_output(['git', 'symbolic-ref', '--short', 'HEAD']).decode().strip())
print('cwd:', os.getcwd())

import camelot
print('camelot version:', camelot.__version__)

## Step 1 — confirm the bug on current master (without the patch)

The xfailed tests should currently FAIL when run with `strict=True`. If they pass on master, the bug was already fixed by some unrelated change and this whole exercise is moot.

In [ ]:
# Run the existing xfailed tests with strict=True to see them fail (= bug still present).
import subprocess
result = subprocess.run(
    ['python', '-m', 'pytest',
     'tests/test_network.py::test_issue_585',
     'tests/test_network.py::test_issue_585_network_flavor_with_table_areas',
     '-v', '--no-header', '-o', 'xfail_strict=true'],
    capture_output=True, text=True,
)
print(result.stdout[-2000:])
print(result.stderr[-500:])

## Step 2 — apply the patch in-memory

Monkey-patch `compute_plausible_gaps` to use the corrected formula. Lets us verify without modifying the source first.

In [ ]:
import numpy as np
from camelot.parsers.network import TextNetworks

_original = TextNetworks.compute_plausible_gaps

def patched_compute_plausible_gaps(self, ref_h_textlines, ref_v_textlines):
    h_textlines = sorted(ref_h_textlines, key=lambda t: t.x0)
    v_textlines = sorted(ref_v_textlines, key=lambda t: t.y0)
    # The change: .x1 instead of .x0 on the previous element.
    h_gaps = np.array([h_textlines[i].x0 - h_textlines[i - 1].x1
                       for i in range(1, len(h_textlines))])
    v_gaps = np.array([v_textlines[i].y0 - v_textlines[i - 1].y1
                       for i in range(1, len(v_textlines))])
    if h_gaps.size == 0 or v_gaps.size == 0:
        return None
    # Rest of the body is identical to the original — call it for the median/percentile logic.
    # If compute_plausible_gaps has more body lines, inspect the source and inline them here.
    return _original.__wrapped__(self, ref_h_textlines, ref_v_textlines) if hasattr(_original, '__wrapped__') else _original(self, ref_h_textlines, ref_v_textlines)

# Inspect the function's signature first; the exact body might need adjustment.
import inspect
print(inspect.getsource(_original))

**Adjustment needed:** inspect the original `compute_plausible_gaps` (printed above) and replace the monkey-patch's body to match — only the two `.x0 - .x0` → `.x0 - .x1` substitutions change.

Once the monkey-patch matches the rest of the body, apply it and re-run the failing tests.

In [ ]:
# Re-run the xfailed tests with the patch applied. They should xpass now.
# (Update the patch body above to match the original's full body before running.)
TextNetworks.compute_plausible_gaps = patched_compute_plausible_gaps

# Direct parse calls — fastest feedback:
import camelot
print('--- multiple_tables.pdf ---')
tables = camelot.read_pdf('tests/files/multiple_tables.pdf', flavor='network',
                          table_areas=['100,700,500,100'],
                          columns=['150,200,250,300,350,400,450,500'])
print(f'tables found: {len(tables)}')
for i, t in enumerate(tables):
    print(f'  table {i}: shape={t.shape}, confidence={t.confidence:.2f}')

print('\n--- good_energy.pdf ---')
tables = camelot.read_pdf('tests/files/good_energy.pdf', flavor='network',
                          table_areas=['46,213,558,180'],
                          columns=['92,159,262,357,454,534'],
                          split_text=True)
print(f'tables found: {len(tables)}')
for i, t in enumerate(tables):
    print(f'  table {i}: shape={t.shape}, confidence={t.confidence:.2f}')

## Step 3 — make the patch real and run the full network test suite

Once Step 2 confirms the two reproducers are fixed, edit `camelot/parsers/network.py:484` and `:490` directly (the TODO comment from PR #771 marks the spot), and run:

```bash
python -m pytest tests/test_network.py tests/test_hybrid.py -v
```

Acceptance: every existing test passes, plus the two xfails flip to xpass (then remove the markers).

In [ ]:
# Convenience: full network + hybrid sweep.
result = subprocess.run(
    ['python', '-m', 'pytest', 'tests/test_network.py', 'tests/test_hybrid.py', '-v', '--no-header'],
    capture_output=True, text=True,
)
print(result.stdout[-3000:])